# Assignment 1 — QANet

**COMP5329 / Deep Learning — University of Sydney, Semester 1 2026**

Run each section in order. Sections 0–1 are one-time setup steps; Sections 2–4 are the main training and evaluation pipeline.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path if your repo is stored elsewhere in Drive.
PROJECT_ROOT = "/content/drive/MyDrive/Comp5329_Assignment1_2026"

In [ ]:

# Install Python dependencies (run once per session)
!pip install -r {PROJECT_ROOT}/requirements.txt -q
!python -m spacy download en

---
## Section 0 — Environment Setup

Mount Google Drive and install dependencies.

In [3]:
PROJECT_ROOT = "E:\Working Area\Comp5329_Assignment1_2026"
#本地运行的位置

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\Admin\AppData\Local\Temp\ipykernel_32940\3064942964.py:1: SyntaxWarning: invalid escape sequence '\W'
  PROJECT_ROOT = "E:\Working Area\Comp5329_Assignment1_2026"


In [4]:
import sys, os

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())

Working directory: E:\Working Area\Comp5329_Assignment1_2026


---
## Section 1 — Download Data *(delete before submitting)*

Downloads the pre-built mini dataset (sampled SQuAD v1.1 train + full dev set,
with GloVe vectors filtered to the mini vocabulary) from GitHub Releases into `_data/`.

> **One-time step.** Once `_data/` exists on your Drive, delete this section before submission.

In [ ]:
from Tools.download import download_mini

download_mini(data_dir="_data")

---
## Section 2 — Preprocess Data *(delete before submitting)*

Tokenises the SQuAD JSON files, builds word/char vocabularies from GloVe, and writes padded index tensors to `_data/`.

> **One-time step.** Once `_data/*.npz` exists on your Drive, delete this section before submission. Re-run only if you change `para_limit`, `ques_limit`, or other shape parameters.

In [ ]:
from Tools.preproc import preprocess

preprocess(
    train_file="_data/squad/train-mini.json",
    dev_file="_data/squad/dev-v1.1.json",
    glove_word_file="_data/glove/glove.mini.txt",
    target_dir="_data",
    para_limit=400,
    ques_limit=50,
)

---
## Section 3 — Train

Trains QANet on SQuAD v1.1 and saves the best checkpoint to `_model/model.pt`.

In [9]:
from TrainTools.train import train

results = train(
    # ── data paths (must match preprocess outputs) ──────────────────────
    train_npz       = "_data/train.npz",
    dev_npz         = "_data/dev.npz",
    word_emb_json   = "_data/word_emb.json",
    char_emb_json   = "_data/char_emb.json",
    train_eval_json = "_data/train_eval.json",
    dev_eval_json   = "_data/dev_eval.json",
    save_dir        = "_model",
    log_dir         = "_log",

    # ── training loop ────────────────────────────────────────────────────
    num_steps  = 200,
    batch_size = 8,
    seed       = 42,

    # ── vanilla recipe: SGD, no scheduler, NLL loss ───────────────────────
    optimizer_name = "sgd",
    scheduler_name = "step",
    loss_name      = "qa_ce",
    init_name=  "kaiming",
    activation= "leaky_relu",
)

print(f"Best F1: {results['best_f1']:.4f}  |  Best EM: {results['best_em']:.4f}")

100%|██████████| 200/200 [00:35<00:00,  5.59it/s]


STEP      200  loss 3607.159696



100%|██████████| 150/150 [00:06<00:00, 21.83it/s]


VALID(train) loss 69.364830  F1 6.974788  EM 0.000000



100%|██████████| 150/150 [00:06<00:00, 22.24it/s]


TEST        loss 68.713446  F1 6.291428  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 6.2914  Best EM: 0.0000
Best F1: 6.2914  |  Best EM: 0.0000


---
## Section 4 - Experiment: Optimizer/Initialization/Activation Grid Search

Runs a small-budget grid search on the preprocessed mini dataset to compare optimizer, initialization, and activation combinations under the same runtime budget.

- Each run uses the same `num_steps`, `checkpoint`, batch size, and evaluation budget.
- `early_stop` is disabled so every combination gets the same number of update steps.
- `scheduler_name` is explicitly fixed to `none` for every run, avoiding scheduler effects in this comparison.
- The search only uses names that already exist in the project's optimizer, initialization, and activation registries.
- The summary reports the top 3 combinations under the current budget.

> This cell only defines the experiment code. Run it only when you want to launch the full search.

In [6]:
from itertools import product
from pathlib import Path
import json
from TrainTools.train import train

# Compare optimizer/initialization/activation combinations with scheduler fixed to none.
SEARCH_OPTIMIZERS = ["sgd", "sgd_momentum", "adam"]
SEARCH_INITIALIZATIONS = ["kaiming", "xavier"]
SEARCH_ACTIVATIONS = ["relu", "leaky_relu"]
FIXED_SCHEDULER = "lambda"
TOP_K = 3

# All combinations share the same training and evaluation budget.
SHARED_PARAMETERS = {
    "num_steps": 100,
    "checkpoint": 50,
    "val_num_batches": 20,
    "test_num_batches": 20,
    "batch_size": 8,
    "seed": 42,
    "early_stop": 9999,
}

# Keep the rest of the setup fixed so the grid isolates optimizer/init/activation effects.
BASE_TRAIN_KWARGS = {
    "train_npz": "_data/train.npz",
    "dev_npz": "_data/dev.npz",
    "word_emb_json": "_data/word_emb.json",
    "char_emb_json": "_data/char_emb.json",
    "train_eval_json": "_data/train_eval.json",
    "dev_eval_json": "_data/dev_eval.json",
    "loss_name": "qa_ce",
    "learning_rate": 1e-3,
    "grad_clip": 5.0,
    "scheduler_name": FIXED_SCHEDULER,
}

grid_results = []

for optimizer_name, init_name, activation in product(
    SEARCH_OPTIMIZERS,
    SEARCH_INITIALIZATIONS,
    SEARCH_ACTIVATIONS,
):
    run_tag = f"{optimizer_name}__{init_name}__{activation}"
    save_dir = f"_model/experiments/{run_tag}"
    log_dir = f"_log/experiments/{run_tag}"

    print(f"\n=== Running {run_tag} ===")

    run_kwargs = {
        **BASE_TRAIN_KWARGS,
        **SHARED_PARAMETERS,
        "optimizer_name": optimizer_name,
        "init_name": init_name,
        "activation": activation,
        "save_dir": save_dir,
        "log_dir": log_dir,
        "ckpt_name": "model.pt",
    }

    result = train(**run_kwargs)

    if not result["history"]:
        raise RuntimeError(f"No checkpoint metrics were recorded for {run_tag}.")

    best_history = max(
        result["history"],
        key=lambda row: (row["dev_f1"], row["dev_em"], -row["dev_loss"]),
    )

    grid_results.append(
        {
            "optimizer": optimizer_name,
            "init_name": init_name,
            "activation": activation,
            "best_dev_f1": best_history["dev_f1"],
            "best_dev_em": best_history["dev_em"],
            "best_dev_loss": best_history["dev_loss"],
            "best_step": best_history["step"],
            "save_dir": save_dir,
            "ckpt_path": result["ckpt_path"],
        }
    )

if not grid_results:
    raise RuntimeError("Grid search produced no results.")

ranked_results = sorted(
    grid_results,
    key=lambda row: (
        -row["best_dev_f1"],
        -row["best_dev_em"],
        row["best_dev_loss"],
        row["optimizer"],
        row["init_name"],
        row["activation"],
    ),
)

header = (
    f"{'rank':<6}"
    f"{'optimizer':<16}"
    f"{'init_name':<14}"
    f"{'activation':<12}"
    f"{'dev_f1':>10}"
    f"{'dev_em':>10}"
    f"{'dev_loss':>12}"
    f"{'step':>8}"
)
print("\n" + header)
print("-" * len(header))

for rank, row in enumerate(ranked_results, start=1):
    print(
        f"{rank:<6}"
        f"{row['optimizer']:<16}"
        f"{row['init_name']:<14}"
        f"{row['activation']:<12}"
        f"{row['best_dev_f1']:>10.4f}"
        f"{row['best_dev_em']:>10.4f}"
        f"{row['best_dev_loss']:>12.6f}"
        f"{row['best_step']:>8}"
    )

top_k_results = ranked_results[:TOP_K]
print(f"\nTop {TOP_K} combinations under the current budget:")
print(json.dumps(top_k_results, indent=2))

summary_dir = Path("_log/experiments")
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / "grid_search_results.json"
summary_path.write_text(json.dumps(ranked_results, indent=2), encoding="utf-8")
print(f"\nSaved ranked results to: {summary_path}")


=== Running sgd__kaiming__relu ===


100%|██████████| 50/50 [00:27<00:00,  1.82it/s]


STEP       50  loss 4487.208086



100%|██████████| 20/20 [00:03<00:00,  5.33it/s]


VALID(train) loss 69.187464  F1 7.072293  EM 0.000000



...

STEP      100  loss 101.053882



100%|██████████| 20/20 [00:03<00:00,  5.27it/s]


VALID(train) loss 16.302758  F1 8.400254  EM 0.000000



100%|██████████| 20/20 [00:03<00:00,  5.26it/s]


TEST        loss 15.722082  F1 9.009377  EM 0.000000

Learning rate: [0.001]
Training finished.  Best F1: 9.0094  Best EM: 0.0000

rank  optimizer       init_name     activation      dev_f1    dev_em    dev_loss    step
----------------------------------------------------------------------------------------
1     adam            xavier        leaky_relu      9.0094    0.0000   15.722082     100
2     adam            xavier        relu            9.0094    0.0000   16.592686     100
3     sgd_momentum    xavier        relu            8.3463    0.0000   20.080500      50
4     sgd_momentum    xavier        leaky_relu      8.3463    0.0000   20.101890      50
5     sgd             xavier        relu            8.3248    0.0000   26.984001     100
6     sgd             xavier        leaky_relu      8.3248    0.0000   27.019555     100
7     sgd             kaiming       leaky_relu      8.1625    0.0000   64.589192     100
8     sgd_momentum    kaiming       leaky_relu      8.0338    0.0000

---
## Section 5 — Evaluate

Loads the saved checkpoint and runs inference on the full dev set.

In [4]:
from EvaluateTools.evaluate import evaluate

metrics = evaluate(
    dev_npz       = "_data/dev.npz",
    word_emb_json = "_data/word_emb.json",
    char_emb_json = "_data/char_emb.json",
    dev_eval_json = "_data/dev_eval.json",
    save_dir      = "_model",
    log_dir       = "_log",
    ckpt_name     = "model.pt",
)

print(f"F1: {metrics['f1']:.4f}  |  EM: {metrics['exact_match']:.4f}  |  Loss: {metrics['loss']:.6f}")

100%|██████████| 1309/1309 [00:53<00:00, 24.31it/s]


TEST  loss 4.827750  F1 7.764030  EM 1.127568
F1: 7.7640  |  EM: 1.1276  |  Loss: 4.827750
